## CNV analysis and figure for WGS and metagenome data

#### CNV duplications: setup, libraries, and directories

In [1]:
suppressPackageStartupMessages({
  library(tidyverse)
  library(readr)
  library(stringr)
  library(vegan)
  library(ggtext)
  library(cowplot)
  library(scales)
})

data_dir <- "../data"
fig_dir  <- "../figs"

source("project_settings.R")

##### Helper functions

In [2]:
numify <- function(x) {
  if (is.numeric(x)) x else suppressWarnings(readr::parse_number(as.character(x)))
}

parse_meta <- function(x){
  xu <- gsub("[[:space:]]+", "", as.character(x))
  xu <- gsub("-", "_", xu)
  tibble(
    Sample_u = xu,
    Evo_orig = case_when(
      str_detect(xu, "^[aA](?:_|$)") ~ "All ancestral",
      str_detect(xu, "^[dD](?:_|$)") ~ "Evolved 1972",
      str_detect(xu, "^[eE](?:_|$)") ~ "Evolved 1977",
      str_detect(xu, "^[fF](?:_|$)") ~ "All evolved",
      TRUE ~ NA_character_
    ),
    AB = case_when(
      str_detect(xu, "(?:^|_)t1(?:_|$)") ~ 5L,
      str_detect(xu, "(?:^|_)t2(?:_|$)") ~ 50L,
      str_detect(xu, "(?:^|_)t3(?:_|$)") ~ 500L,
      str_detect(xu, "(?:^|_)t4(?:_|$)") ~ 0L,
      TRUE ~ NA_integer_
    ),
    Replicate = case_when(
      str_detect(xu, "(?:^|_)1(?:_|$)") ~ 1L,
      str_detect(xu, "(?:^|_)2(?:_|$)") ~ 2L,
      str_detect(xu, "(?:^|_)3(?:_|$)") ~ 3L,
      str_detect(xu, "(?:^|_)4(?:_|$)") ~ 4L,
      TRUE ~ NA_integer_
    )
  )
}

label_ugml <- function(x) paste0(x, " \u00B5g ml\u207B\u00B9")
hambi_to_strainID <- function(h) sprintf("HAMBI_%04d", as.integer(h))

#### WGS CNV analysis — SIMPLE metrics, absolute QC, and ancestral duplications

##### Fixed thresholds and QC parameters for WGS

In [3]:
MIN_COVERAGE    <- 100
MIN_COPY_NUMBER <- 2.0
MIN_LENGTH_BP   <- 800
MIN_DUP_READS   <- 2

NEAR2_WINDOW      <- 0.15
NEAR2_PROP_MIN    <- 0.80
NEAR2_MIN_GENES   <- 80

##### Read WGS CNV data

In [4]:
wgs_cnv <- read.delim(
  file.path(data_dir, "wgs_cnv_calls.tsv"),
  check.names = FALSE,
  stringsAsFactors = FALSE
)

##### Numeric parsing for WGS

In [5]:
wgs_num_cols <- c(
  "LENGTH","COVERAGE","COPY_NUMBER",
  "reads_supporting_Deletion",
  "reads_supporting_Tandem_Direct","reads_supporting_Tandem_Indirect",
  "reads_supporting_Interspersed_Direct","reads_supporting_Interspersed_Indirect"
)

wgs_present_num_cols <- intersect(wgs_num_cols, names(wgs_cnv))
wgs_cnv <- wgs_cnv %>% mutate(across(all_of(wgs_present_num_cols), numify))

##### Require essential WGS columns

In [6]:
wgs_req_cols <- c("TYPE","COPY_NUMBER","COVERAGE","LENGTH","HAMBI","ID")
wgs_miss <- setdiff(wgs_req_cols, names(wgs_cnv))
if (length(wgs_miss) > 0) {
  stop("Missing required columns in WGS CNV table: ", paste(wgs_miss, collapse = ", "))
}

##### Duplication support for WGS

In [7]:
wgs_dup_cols <- intersect(
  c("reads_supporting_Tandem_Direct","reads_supporting_Tandem_Indirect",
    "reads_supporting_Interspersed_Direct","reads_supporting_Interspersed_Indirect"),
  names(wgs_cnv)
)

if (length(wgs_dup_cols) == 0) {
  stop("Duplication-support columns are missing in WGS data; cannot enforce MIN_DUP_READS.")
}

wgs_cnv <- wgs_cnv %>%
  mutate(dup_reads_support = do.call(pmax, c(across(all_of(wgs_dup_cols)), na.rm = TRUE)))

##### Species label cleanup and drop NA or blank IDs

In [8]:
wgs_cnv <- wgs_cnv %>%
  mutate(
    Species = sub("^(HAMBI_\\d+).*", "\\1", as.character(HAMBI)),
    ID      = as.character(ID)
  ) %>%
  filter(!is.na(ID), ID != "", ID != "NA")

##### SIMPLE row-level filtering for WGS

In [9]:
wgs_cnv_kept_rows <- wgs_cnv %>%
  filter(tolower(TYPE) != "deletion") %>%
  filter(!is.na(COVERAGE)    & COVERAGE    >= MIN_COVERAGE) %>%
  filter(!is.na(COPY_NUMBER) & COPY_NUMBER >= MIN_COPY_NUMBER) %>%
  filter(!is.na(LENGTH)      & LENGTH      >= MIN_LENGTH_BP) %>%
  filter(!is.na(dup_reads_support) & dup_reads_support >= MIN_DUP_READS)

##### Absolute species-level QC for WGS

In [10]:
wgs_species_diag <- wgs_cnv_kept_rows %>%
  group_by(Species) %>%
  summarise(
    dup_genes  = n_distinct(ID),
    near2_prop = mean(abs(COPY_NUMBER - 2) <= NEAR2_WINDOW, na.rm = TRUE),
    .groups    = "drop"
  )

wgs_species_to_drop <- wgs_species_diag %>%
  filter(dup_genes >= NEAR2_MIN_GENES & near2_prop >= NEAR2_PROP_MIN) %>%
  arrange(desc(dup_genes))

wgs_cnv_kept <- wgs_cnv_kept_rows %>%
  filter(!(Species %in% wgs_species_to_drop$Species))

##### Summaries after WGS QC

In [11]:
wgs_summary_recurrent <- wgs_cnv_kept %>%
  mutate(
    Gene = if ("gene_name" %in% names(.)) {
      ifelse(!is.na(gene_name) & gene_name != "", gene_name, ID)
    } else {
      ID
    }
  ) %>%
  group_by(Species, Gene) %>%
  summarise(
    n_events           = dplyr::n(),
    copy_number_median = suppressWarnings(median(COPY_NUMBER, na.rm = TRUE)),
    .groups = "drop"
  ) %>%
  arrange(Species, desc(copy_number_median), Gene)

wgs_dup_counts_after <- wgs_summary_recurrent %>%
  group_by(Species) %>%
  summarise(duplicated_genes = n_distinct(Gene), .groups = "drop") %>%
  arrange(desc(duplicated_genes))

##### WGS stats output

In [12]:
cat("\n=== WGS duplicated genes per species (after QC) ===\n")
print(wgs_dup_counts_after, n = Inf)


=== WGS duplicated genes per species (after QC) ===
# A tibble: 4 × 2
  Species    duplicated_genes
  <chr>                 <int>
1 HAMBI_1972               31
2 HAMBI_1896               20
3 HAMBI_2659               14
4 HAMBI_1977                2


##### Ancestral duplications object from WGS (in memory only)

In [13]:
ancestral_duplications_wgs <- wgs_cnv_kept %>%
  transmute(
    Species,
    ID,
    Gene = if ("gene_name" %in% names(.)) {
      ifelse(!is.na(gene_name) & gene_name != "", gene_name, ID)
    } else {
      ID
    }
  ) %>%
  distinct() %>%
  arrange(Species, ID)

#### Metagenome CNV duplications — QC and removal of WGS ancestral dups

##### Read input metagenome CNV table

In [14]:
cnv <- read.delim(
  file.path(data_dir, "metagenome_cnv_calls.tsv"),
  check.names = FALSE,
  stringsAsFactors = FALSE
)

##### Remove CC controls from metagenome data

In [15]:
cnv <- cnv %>% filter(toupper(Sample) != "CC")

##### Parse numeric columns and keep duplication events in metagenomes

In [16]:
meta_num_cols <- c(
  "START.CNV","END.CNV","LENGTH","COVERAGE","COPY_NUMBER","reads_TOTAL",
  "reads_supporting_Tandem_Direct","reads_supporting_Tandem_Indirect",
  "reads_supporting_Interspersed_Direct","reads_supporting_Interspersed_Indirect",
  "START.GENE","END.GENE"
)

In [17]:
meta_present_num_cols <- intersect(meta_num_cols, names(cnv))
cnv <- cnv %>% mutate(across(all_of(meta_present_num_cols), numify))

stopifnot("TYPE" %in% names(cnv))
cnv <- cnv %>% filter(tolower(TYPE) %in% "dup")

##### Compute duplication support in metagenomes

In [18]:
meta_dup_cols <- intersect(
  c("reads_supporting_Tandem_Direct","reads_supporting_Tandem_Indirect",
    "reads_supporting_Interspersed_Direct","reads_supporting_Interspersed_Indirect"),
  names(cnv)
)

if (length(meta_dup_cols) == 0) {
  stop("No duplication-support columns found in metagenome data.")
}

cnv <- cnv %>%
  mutate(dup_reads_support = do.call(pmax, c(across(all_of(meta_dup_cols)), na.rm = TRUE)))

#### Metagenome QC filters, gene coverage, and removal of ancestral WGS dups

##### Conservative QC and gene coverage fraction for metagenomes

In [19]:
meta_req_cols <- c(
  "COPY_NUMBER","LENGTH","dup_reads_support","reads_TOTAL",
  "gene_name","START.GENE","END.GENE","START.CNV","END.CNV",
  "HAMBI","Sample"
)

meta_miss <- setdiff(meta_req_cols, names(cnv))
if (length(meta_miss)) {
  stop("Missing required columns in metagenome CNV table: ", paste(meta_miss, collapse = ", "))
}

cnv_filt <- cnv %>%
  filter(
    COPY_NUMBER >= 2,
    LENGTH >= 300,
    dup_reads_support >= 5,
    reads_TOTAL >= 20,
    !is.na(gene_name), gene_name != "",
    !is.na(START.GENE), !is.na(END.GENE),
    !is.na(START.CNV),  !is.na(END.CNV)
  ) %>%
  rowwise() %>%
  mutate(
    gene_cov_frac = {
      s_gene <- min(START.GENE, END.GENE, na.rm = TRUE)
      e_gene <- max(START.GENE, END.GENE, na.rm = TRUE)
      s_cnv  <- min(START.CNV,  END.CNV,  na.rm = TRUE)
      e_cnv  <- max(START.CNV,  END.CNV,  na.rm = TRUE)
      left   <- max(s_gene, s_cnv)
      right  <- min(e_gene, e_cnv)
      overlap <- max(0, right - left + 1)
      gene_len <- max(1, e_gene - s_gene + 1)
      overlap / gene_len
    }
  ) %>%
  ungroup() %>%
  filter(gene_cov_frac >= 0.5) %>%
  mutate(
    Species = HAMBI,
    Gene    = gene_name
  )

##### Remove ancestral WGS duplications from filtered metagenome CNVs

In [20]:
if (!exists("ancestral_duplications_wgs")) {
  stop("ancestral_duplications_wgs object not found; ensure WGS block ran before metagenome block.")
}

anc_norm <- ancestral_duplications_wgs %>%
  mutate(
    Species_base = sub("^(HAMBI_\\d+).*", "\\1", as.character(Species)),
    Gene_key     = tolower(trimws(as.character(dplyr::coalesce(Gene, ID)))),
    ID_key       = tolower(trimws(as.character(ID)))
  )

anc_gene <- anc_norm %>%
  filter(!is.na(Gene_key), Gene_key != "") %>%
  distinct(Species_base, Gene_key)

anc_id <- anc_norm %>%
  filter(!is.na(ID_key), ID_key != "") %>%
  distinct(Species_base, ID_key)

cnv_filt_anc <- cnv_filt %>%
  mutate(
    Species_base = sub("^(HAMBI_\\d+).*", "\\1", as.character(Species)),
    Gene_key     = tolower(trimws(as.character(Gene))),
    ID_key       = if ("ID" %in% names(.)) tolower(trimws(as.character(ID))) else NA_character_
  )

if (nrow(anc_gene) > 0) {
  cnv_filt_anc <- cnv_filt_anc %>%
    anti_join(anc_gene, by = c("Species_base" = "Species_base",
                               "Gene_key"     = "Gene_key"))
}

if ("ID" %in% names(cnv_filt_anc) && nrow(anc_id) > 0) {
  cnv_filt_anc <- cnv_filt_anc %>%
    anti_join(anc_id, by = c("Species_base" = "Species_base",
                             "ID_key"       = "ID_key"))
}

cnv_filt_anc <- cnv_filt_anc %>%
  select(-any_of(c("Species_base","Gene_key","ID_key")))

##### Recurrent metagenome targets after ancestral filtering (present in at least 2 samples)

In [21]:
recur_pairs <- cnv_filt_anc %>%
  group_by(Species, Gene) %>%
  summarise(n_samples = n_distinct(Sample), .groups = "drop") %>%
  filter(n_samples >= 2)

if (nrow(recur_pairs) == 0) {
  cat("\n=== Metagenome CNVdup PERMANOVA (Jaccard) ===\n")
  cat("NOTE: No recurrent targets (>=2 samples) after QC and ancestral filtering.\n")
  cnv_metagenome_dup_heatmap <- ggplot() + theme_void() +
    annotate("text", x = .5, y = .55,
             label = "No recurrent duplicated genes after ancestral filtering.",
             size = 5, fontface = "bold")
  pdf_dev <- if (capabilities("cairo")) cairo_pdf else "pdf"
  ggsave(
    filename = file.path(fig_dir, "ED_Fig6.pdf"),
    plot     = cnv_metagenome_dup_heatmap,
    width    = 8,
    height   = 5.5,
    units    = "in",
    device   = pdf_dev
  )
  stop("No recurrent targets after ancestral filtering.")
}

cnv_recur <- cnv_filt_anc %>%
  inner_join(recur_pairs %>% select(Species, Gene), by = c("Species","Gene"))

#### Build metagenome presence-absence matrix and sample metadata

##### Parse metadata from metagenome sample names

In [22]:
cnv_std <- cnv_recur %>%
  bind_cols(parse_meta(.$Sample)) %>%
  mutate(
    strainID = sprintf("HAMBI_%04d", as.integer(gsub("\\D","", Species)))
  ) %>%
  filter(!is.na(Evo_orig), !is.na(AB))

if (nrow(cnv_std) == 0) {
  wcat("\n=== Metagenome CNVdup PERMANOVA (Jaccard) ===\n")
  cat("NOTE: No parseable rows after metadata parsing.\n")
  cnv_metagenome_dup_heatmap <- ggplot() + theme_void() +
    annotate("text", x = .5, y = .55,
             label = "No parseable recurrent CNV rows after ancestral filtering.",
             size = 5, fontface = "bold")
  pdf_dev <- if (capabilities("cairo")) cairo_pdf else "pdf"
  ggsave(
    filename = file.path(fig_dir, "ED_Fig6.pdf"),
    plot     = cnv_metagenome_dup_heatmap,
    width    = 8,
    height   = 5.5,
    units    = "in",
    device   = pdf_dev
  )
  stop("No parseable rows after metadata parsing in metagenome data.")
}

##### Metagenome presence-absence matrix over recurrent targets

In [23]:
cnv_bin <- cnv_std %>%
  transmute(
    Sample_u,
    Evo_orig,
    AB,
    TargetID = paste(strainID, Gene, sep = "|"),
    value    = 1L
  )

mat_wide <- cnv_bin %>%
  tidyr::pivot_wider(
    names_from  = TargetID,
    values_from = value,
    values_fill = 0,
    values_fn   = max
  )

meta <- mat_wide %>% select(Sample_u, Evo_orig, AB)
Y_df <- mat_wide %>% select(-Sample_u, -Evo_orig, -AB)

##### Ensure numeric matrix and drop all-zero rows and columns

In [24]:
Y_df <- Y_df %>%
  mutate(across(
    everything(),
    ~ {
      if (is.list(.)) as.numeric(lengths(.) > 0)
      else if (is.logical(.)) as.numeric(.)
      else suppressWarnings(as.numeric(.))
    }
  )) %>%
  replace(is.na(.), 0)

Y <- as.matrix(Y_df)

keep_cols <- which(colSums(Y, na.rm = TRUE) > 0)
if (length(keep_cols) == 0) {
  cat("\n=== Metagenome CNVdup PERMANOVA (Jaccard) ===\n")
  cat("NOTE: All targets are zero across samples after ancestral filtering.\n")
  cnv_metagenome_dup_heatmap <- ggplot() + theme_void() +
    annotate("text", x = .5, y = .55,
             label = "All-zero target matrix after ancestral filtering.",
             size = 5, fontface = "bold")
  pdf_dev <- if (capabilities("cairo")) cairo_pdf else "pdf"
  ggsave(
    filename = file.path(fig_dir, "ED_Fig6.pdf"),
    plot     = cnv_metagenome_dup_heatmap,
    width    = 8,
    height   = 5.5,
    units    = "in",
    device   = pdf_dev
  )
  stop("All-zero target matrix after ancestral filtering.")
}
Y <- Y[, keep_cols, drop = FALSE]

keep_rows <- which(rowSums(Y, na.rm = TRUE) > 0)
if (length(keep_rows) < 2) {
  cat("\n=== Metagenome CNVdup PERMANOVA (Jaccard) ===\n")
  cat("NOTE: Too few non-zero samples after cleaning.\n")
  cnv_metagenome_dup_heatmap <- ggplot() + theme_void() +
    annotate("text", x = .5, y = .55,
             label = "Too few non-zero samples for PERMANOVA.",
             size = 5, fontface = "bold")
  pdf_dev <- if (capabilities("cairo")) cairo_pdf else "pdf"
  ggsave(
    filename = file.path(fig_dir, "ED_Fig6.pdf"),
    plot     = cnv_metagenome_dup_heatmap,
    width    = 8,
    height   = 5.5,
    units    = "in",
    device   = pdf_dev
  )
  stop("Too few non-zero samples after cleaning in metagenome matrix.")
}

Y    <- Y[keep_rows, , drop = FALSE]
meta <- meta[keep_rows, , drop = FALSE]

#### PERMANOVA on metagenome CNV duplications (Jaccard, mutation-style formula)

##### Prepare data frame for PERMANOVA

In [25]:
meta_permanova <- meta %>%
  mutate(
    Primer_pulse = factor(AB, levels = c(0,5,50,500)),
    Evo          = factor(
      Evo_orig,
      levels = c("All ancestral","Evolved 1977","Evolved 1972","All evolved")
    )
  ) %>%
  select(sample = Sample_u, Primer_pulse, Evo)

gene_table <- meta_permanova %>%
  bind_cols(as.data.frame(Y))

##### Run PERMANOVA: response = gene_table[,-(1:3)], predictors from gene_table[,1:3]

In [26]:
permanova <- adonis2(
  gene_table[, -(1:3)] ~ Primer_pulse * Evo,
  data        = gene_table[, 1:3],
  permutations = 9999,
  method       = "jaccard",
  binary       = TRUE,
  na.rm        = TRUE,
  by           = "terms"
)

permanova_df <- as.data.frame(permanova)
permanova_df$Factor <- rownames(permanova_df)
rownames(permanova_df) <- NULL
print(permanova)

Permutation test for adonis under reduced model
Terms added sequentially (first to last)
Permutation: free
Number of permutations: 9999

adonis2(formula = gene_table[, -(1:3)] ~ Primer_pulse * Evo, data = gene_table[, 1:3], permutations = 9999, method = "jaccard", by = "terms", binary = TRUE, na.rm = TRUE)
                 Df SumOfSqs      R2      F Pr(>F)    
Primer_pulse      3   1.4138 0.17583 4.3112 0.0001 ***
Evo               3   1.4572 0.18123 4.4436 0.0001 ***
Primer_pulse:Evo  7   1.0159 0.12635 1.3277 0.1448    
Residual         38   4.1539 0.51660                  
Total            51   8.0407 1.00000                  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


##### Save PERMANOVA results

In [27]:
cat("\n=== PERMANOVA (Jaccard, CNV metagenome) ===\n")
print(permanova_df[, c("Factor", setdiff(names(permanova_df), "Factor"))], row.names = FALSE)

cat("\n=== Levels summary ===\n")
print(tibble(
  N_samples       = nrow(Y),
  N_targets       = ncol(Y),
  Levels_Primer   = nlevels(meta_permanova$Primer_pulse),
  Levels_Evo      = nlevels(meta_permanova$Evo)
))


=== PERMANOVA (Jaccard, CNV metagenome) ===
           Factor Df SumOfSqs        R2        F Pr(>F)
     Primer_pulse  3 1.413782 0.1758271 4.311153 0.0001
              Evo  3 1.457200 0.1812269 4.443552 0.0001
 Primer_pulse:Evo  7 1.015913 0.1263456 1.327672 0.1448
         Residual 38 4.153854 0.5166004       NA     NA
            Total 51 8.040748 1.0000000       NA     NA

=== Levels summary ===
# A tibble: 1 × 4
  N_samples N_targets Levels_Primer Levels_Evo
      <int>     <int>         <int>      <int>
1        52         9             4          4


#### Recurrent CNV duplications — metagenome heatmap for ED_Fig6

In [28]:
#!/usr/bin/env Rscript

suppressPackageStartupMessages({
  library(tidyverse)
  library(readr)
  library(stringr)
  library(vegan)
  library(ggtext)
  library(cowplot)
  library(scales)
})

source("project_settings.R")

numify <- function(x) {
  if (is.numeric(x)) x else suppressWarnings(readr::parse_number(as.character(x)))
}

parse_meta <- function(x){
  xu <- gsub("[[:space:]]+", "", as.character(x))
  xu <- gsub("-", "_", xu)
  tibble(
    Sample_u = xu,
    Evo_orig = case_when(
      str_detect(xu, "^[aA](?:_|$)") ~ "All ancestral",
      str_detect(xu, "^[dD](?:_|$)") ~ "Evolved 1972",
      str_detect(xu, "^[eE](?:_|$)") ~ "Evolved 1977",
      str_detect(xu, "^[fF](?:_|$)") ~ "All evolved",
      TRUE ~ NA_character_
    ),
    AB = case_when(
      str_detect(xu, "(?:^|_)t1(?:_|$)") ~ 5L,
      str_detect(xu, "(?:^|_)t2(?:_|$)") ~ 50L,
      str_detect(xu, "(?:^|_)t3(?:_|$)") ~ 500L,
      str_detect(xu, "(?:^|_)t4(?:_|$)") ~ 0L,
      TRUE ~ NA_integer_
    ),
    Replicate = case_when(
      str_detect(xu, "(?:^|_)1(?:_|$)") ~ 1L,
      str_detect(xu, "(?:^|_)2(?:_|$)") ~ 2L,
      str_detect(xu, "(?:^|_)3(?:_|$)") ~ 3L,
      str_detect(xu, "(?:^|_)4(?:_|$)") ~ 4L,
      TRUE ~ NA_integer_
    )
  )
}

hambi_to_strainID <- function(h) sprintf("HAMBI_%04d", as.integer(h))
hambi_to_strainID_dash <- function(h) sprintf("HAMBI-%04d", as.integer(h))

abbr_taxon <- function(genus, species) {
  ifelse(
    is.na(genus) | is.na(species) | genus == "" | species == "",
    NA_character_,
    paste0(substr(genus, 1, 1), ". ", species)
  )
}

# ---- robust plotmath label constructor (VECTORISED) ----
# returns a character vector of plotmath expression strings
to_plotmath_tax_gene <- function(tax_abbr, gene_core) {
  tax_abbr  <- as.character(tax_abbr)
  gene_core <- as.character(gene_core)

  esc <- function(s) {
    s <- ifelse(is.na(s), "", s)
    s <- gsub("\\\\", "\\\\\\\\", s)
    s <- gsub("'", "\\\\'", s)
    s
  }

  ta <- esc(tax_abbr)
  gc <- esc(gene_core)

  has_tax  <- !is.na(tax_abbr)  & tax_abbr  != ""
  has_gene <- !is.na(gene_core) & gene_core != ""

  out <- character(length(tax_abbr))
  out[ has_tax &  has_gene] <- paste0("italic('", ta[has_tax & has_gene], "')~'", gc[has_tax & has_gene], "'")
  out[ has_tax & !has_gene] <- paste0("italic('", ta[has_tax & !has_gene], "')")
  out[!has_tax &  has_gene] <- paste0("'", gc[!has_tax & has_gene], "'")
  out[!has_tax & !has_gene] <- "''"

  out
}

# ==========================================================
# -------------------- DATA PROCESSING (UNCHANGED) --------------------
# ==========================================================

MIN_COVERAGE    <- 100
MIN_COPY_NUMBER <- 2.0
MIN_LENGTH_BP   <- 800
MIN_DUP_READS   <- 2

NEAR2_WINDOW      <- 0.15
NEAR2_PROP_MIN    <- 0.80
NEAR2_MIN_GENES   <- 80

wgs_cnv <- read.delim(
  file.path(data_dir, "wgs_cnv_calls.tsv"),
  check.names = FALSE,
  stringsAsFactors = FALSE
)

wgs_num_cols <- c(
  "LENGTH","COVERAGE","COPY_NUMBER",
  "reads_supporting_Deletion",
  "reads_supporting_Tandem_Direct","reads_supporting_Tandem_Indirect",
  "reads_supporting_Interspersed_Direct","reads_supporting_Interspersed_Indirect"
)

wgs_present_num_cols <- intersect(wgs_num_cols, names(wgs_cnv))
wgs_cnv <- wgs_cnv %>% mutate(across(all_of(wgs_present_num_cols), numify))

wgs_req_cols <- c("TYPE","COPY_NUMBER","COVERAGE","LENGTH","HAMBI","ID")
wgs_miss <- setdiff(wgs_req_cols, names(wgs_cnv))
if (length(wgs_miss) > 0) {
  stop("Missing required columns in WGS CNV table: ", paste(wgs_miss, collapse = ", "))
}

wgs_dup_cols <- intersect(
  c("reads_supporting_Tandem_Direct","reads_supporting_Tandem_Indirect",
    "reads_supporting_Interspersed_Direct","reads_supporting_Interspersed_Indirect"),
  names(wgs_cnv)
)

if (length(wgs_dup_cols) == 0) {
  stop("Duplication-support columns are missing in WGS data; cannot enforce MIN_DUP_READS.")
}

wgs_cnv <- wgs_cnv %>%
  mutate(dup_reads_support = do.call(pmax, c(across(all_of(wgs_dup_cols)), na.rm = TRUE)))

wgs_cnv <- wgs_cnv %>%
  mutate(
    Species = sub("^(HAMBI_\\d+).*", "\\1", as.character(HAMBI)),
    ID      = as.character(ID)
  ) %>%
  filter(!is.na(ID), ID != "", ID != "NA")

wgs_cnv_kept_rows <- wgs_cnv %>%
  filter(tolower(TYPE) != "deletion") %>%
  filter(!is.na(COVERAGE)    & COVERAGE    >= MIN_COVERAGE) %>%
  filter(!is.na(COPY_NUMBER) & COPY_NUMBER >= MIN_COPY_NUMBER) %>%
  filter(!is.na(LENGTH)      & LENGTH      >= MIN_LENGTH_BP) %>%
  filter(!is.na(dup_reads_support) & dup_reads_support >= MIN_DUP_READS)

wgs_species_diag <- wgs_cnv_kept_rows %>%
  group_by(Species) %>%
  summarise(
    dup_genes  = n_distinct(ID),
    near2_prop = mean(abs(COPY_NUMBER - 2) <= NEAR2_WINDOW, na.rm = TRUE),
    .groups    = "drop"
  )

wgs_species_to_drop <- wgs_species_diag %>%
  filter(dup_genes >= NEAR2_MIN_GENES & near2_prop >= NEAR2_PROP_MIN) %>%
  arrange(desc(dup_genes))

wgs_cnv_kept <- wgs_cnv_kept_rows %>%
  filter(!(Species %in% wgs_species_to_drop$Species))

wgs_summary_recurrent <- wgs_cnv_kept %>%
  mutate(
    Gene = if ("gene_name" %in% names(.)) {
      ifelse(!is.na(gene_name) & gene_name != "", gene_name, ID)
    } else {
      ID
    }
  ) %>%
  group_by(Species, Gene) %>%
  summarise(
    n_events           = dplyr::n(),
    copy_number_median = suppressWarnings(median(COPY_NUMBER, na.rm = TRUE)),
    .groups = "drop"
  ) %>%
  arrange(Species, desc(copy_number_median), Gene)

wgs_dup_counts_after <- wgs_summary_recurrent %>%
  group_by(Species) %>%
  summarise(duplicated_genes = n_distinct(Gene), .groups = "drop") %>%
  arrange(desc(duplicated_genes))

cat("\n=== WGS duplicated genes per species (after QC) ===\n")
print(wgs_dup_counts_after, n = Inf)

ancestral_duplications_wgs <- wgs_cnv_kept %>%
  transmute(
    Species,
    ID,
    Gene = if ("gene_name" %in% names(.)) {
      ifelse(!is.na(gene_name) & gene_name != "", gene_name, ID)
    } else {
      ID
    }
  ) %>%
  distinct() %>%
  arrange(Species, ID)

cnv <- read.delim(
  file.path(data_dir, "metagenome_cnv_calls.tsv"),
  check.names = FALSE,
  stringsAsFactors = FALSE
)

cnv <- cnv %>% filter(toupper(Sample) != "CC")

meta_num_cols <- c(
  "START.CNV","END.CNV","LENGTH","COVERAGE","COPY_NUMBER","reads_TOTAL",
  "reads_supporting_Tandem_Direct","reads_supporting_Tandem_Indirect",
  "reads_supporting_Interspersed_Direct","reads_supporting_Interspersed_Indirect",
  "START.GENE","END.GENE"
)

meta_present_num_cols <- intersect(meta_num_cols, names(cnv))
cnv <- cnv %>% mutate(across(all_of(meta_present_num_cols), numify))

stopifnot("TYPE" %in% names(cnv))
cnv <- cnv %>% filter(tolower(TYPE) %in% "dup")

meta_dup_cols <- intersect(
  c("reads_supporting_Tandem_Direct","reads_supporting_Tandem_Indirect",
    "reads_supporting_Interspersed_Direct","reads_supporting_Interspersed_Indirect"),
  names(cnv)
)

if (length(meta_dup_cols) == 0) {
  stop("No duplication-support columns found in metagenome data.")
}

cnv <- cnv %>%
  mutate(dup_reads_support = do.call(pmax, c(across(all_of(meta_dup_cols)), na.rm = TRUE)))

meta_req_cols <- c(
  "COPY_NUMBER","LENGTH","dup_reads_support","reads_TOTAL",
  "gene_name","START.GENE","END.GENE","START.CNV","END.CNV",
  "HAMBI","Sample"
)

meta_miss <- setdiff(meta_req_cols, names(cnv))
if (length(meta_miss)) {
  stop("Missing required columns in metagenome CNV table: ", paste(meta_miss, collapse = ", "))
}

cnv_filt <- cnv %>%
  filter(
    COPY_NUMBER >= 2,
    LENGTH >= 300,
    dup_reads_support >= 5,
    reads_TOTAL >= 20,
    !is.na(gene_name), gene_name != "",
    !is.na(START.GENE), !is.na(END.GENE),
    !is.na(START.CNV),  !is.na(END.CNV)
  ) %>%
  rowwise() %>%
  mutate(
    gene_cov_frac = {
      s_gene <- min(START.GENE, END.GENE, na.rm = TRUE)
      e_gene <- max(START.GENE, END.GENE, na.rm = TRUE)
      s_cnv  <- min(START.CNV,  END.CNV,  na.rm = TRUE)
      e_cnv  <- max(START.CNV,  END.CNV,  na.rm = TRUE)
      left   <- max(s_gene, s_cnv)
      right  <- min(e_gene, e_cnv)
      overlap <- max(0, right - left + 1)
      gene_len <- max(1, e_gene - s_gene + 1)
      overlap / gene_len
    }
  ) %>%
  ungroup() %>%
  filter(gene_cov_frac >= 0.5) %>%
  mutate(
    Species = HAMBI,
    Gene    = gene_name
  )

anc_norm <- ancestral_duplications_wgs %>%
  mutate(
    Species_base = sub("^(HAMBI_\\d+).*", "\\1", as.character(Species)),
    Gene_key     = tolower(trimws(as.character(dplyr::coalesce(Gene, ID)))),
    ID_key       = tolower(trimws(as.character(ID)))
  )

anc_gene <- anc_norm %>%
  filter(!is.na(Gene_key), Gene_key != "") %>%
  distinct(Species_base, Gene_key)

anc_id <- anc_norm %>%
  filter(!is.na(ID_key), ID_key != "") %>%
  distinct(Species_base, ID_key)

cnv_filt_anc <- cnv_filt %>%
  mutate(
    Species_base = sub("^(HAMBI_\\d+).*", "\\1", as.character(Species)),
    Gene_key     = tolower(trimws(as.character(Gene))),
    ID_key       = if ("ID" %in% names(.)) tolower(trimws(as.character(ID))) else NA_character_
  )

if (nrow(anc_gene) > 0) {
  cnv_filt_anc <- cnv_filt_anc %>%
    anti_join(anc_gene, by = c("Species_base" = "Species_base",
                               "Gene_key"     = "Gene_key"))
}

if ("ID" %in% names(cnv_filt_anc) && nrow(anc_id) > 0) {
  cnv_filt_anc <- cnv_filt_anc %>%
    anti_join(anc_id, by = c("Species_base" = "Species_base",
                             "ID_key"       = "ID_key"))
}

cnv_filt_anc <- cnv_filt_anc %>%
  select(-any_of(c("Species_base","Gene_key","ID_key")))

recur_pairs <- cnv_filt_anc %>%
  group_by(Species, Gene) %>%
  summarise(n_samples = n_distinct(Sample), .groups = "drop") %>%
  filter(n_samples >= 2)

if (nrow(recur_pairs) == 0) {
  cat("\n=== Metagenome CNVdup PERMANOVA (Jaccard) ===\n")
  cat("NOTE: No recurrent targets (>=2 samples) after QC and ancestral filtering.\n")
  cnv_metagenome_dup_heatmap <- ggplot() + theme_void() +
    annotate("text", x = .5, y = .55,
             label = "No recurrent duplicated genes after ancestral filtering.",
             size = 5, fontface = "bold")
  pdf_dev <- if (capabilities("cairo")) cairo_pdf else "pdf"
  ggsave(file.path(fig_dir, "ED_Fig6.pdf"), cnv_metagenome_dup_heatmap,
         width = 5.2, height = 5.5, units = "in", device = pdf_dev)
  stop("No recurrent targets after ancestral filtering.")
}

cnv_recur <- cnv_filt_anc %>%
  inner_join(recur_pairs %>% select(Species, Gene), by = c("Species","Gene"))

cnv_meta_plot <- cnv_recur %>%
  bind_cols(parse_meta(.$Sample)) %>%
  mutate(
    HAMBI_num = suppressWarnings(as.integer(gsub("\\D","", as.character(Species)))),
    strainID  = hambi_to_strainID(HAMBI_num),
    strainID_dash = hambi_to_strainID_dash(HAMBI_num),
    AB_fac    = factor(AB, levels = c(0,5,50,500), labels = c("0","5","50","500")),
    Evo_fac   = factor(
      Evo_orig,
      levels = c("All ancestral","Evolved 1972","Evolved 1977","All evolved"),
      labels = c("Anc. comm.", "Dom. species\nprimed", "Subdom. species\nprimed", "Full primed")
    ),
    rep_id = ifelse(is.na(Replicate), Sample_u, as.character(Replicate))
  ) %>%
  filter(!is.na(Evo_orig), !is.na(AB))

cnv_bin_plot <- cnv_meta_plot %>%
  distinct(Sample_u, Evo_fac, AB, AB_fac, rep_id, strainID, strainID_dash, Gene) %>%
  mutate(present = 1L)

hf_dup <- cnv_bin_plot %>%
  group_by(strainID, strainID_dash, Gene, Evo_fac, AB_fac) %>%
  summarise(COUNT = n_distinct(rep_id[present == 1L]), .groups = "drop") %>%
  mutate(gene_chr = as.character(Gene))

# ==========================================================
# -------------------- FIGURE SETTINGS ONLY ----------------
# ==========================================================

# taxonomy join
if (exists("tax") && all(c("strainID","genus","species") %in% names(tax))) {
  hf_dup <- hf_dup %>%
    left_join(tax %>% select(strainID, genus, species),
              by = c("strainID_dash" = "strainID")) %>%
    mutate(tax_abbr = abbr_taxon(genus, species))
} else {
  hf_dup <- hf_dup %>% mutate(tax_abbr = NA_character_)
}

hf_dup <- hf_dup %>%
  mutate(
    gene_core = ifelse(!is.na(gene_chr) & gene_chr != "", gene_chr, ""),
    row_label_plain = case_when(
      !is.na(tax_abbr) & tax_abbr != "" & gene_core != "" ~ paste0(tax_abbr, " ", gene_core),
      !is.na(tax_abbr) & tax_abbr != ""                   ~ tax_abbr,
      gene_core != ""                                     ~ gene_core,
      TRUE                                                ~ strainID
    ),
    row_label_expr = to_plotmath_tax_gene(tax_abbr, gene_core)
  )

ord_tbl <- hf_dup %>%
  group_by(strainID, row_label_plain) %>%
  summarise(maxCOUNT = suppressWarnings(max(COUNT, na.rm = TRUE)), .groups = "drop") %>%
  mutate(maxCOUNT = ifelse(is.infinite(maxCOUNT) | is.na(maxCOUNT), 0, maxCOUNT)) %>%
  arrange(strainID, maxCOUNT, row_label_plain)

y_levels <- unique(ord_tbl$row_label_plain) %>% rev()

y_expr_map <- hf_dup %>%
  transmute(row_label_plain, row_label_expr) %>%
  distinct() %>%
  deframe()

y_num_tbl <- tibble(row_label_plain = y_levels) %>% mutate(y_num = row_number())
y_max <- max(y_num_tbl$y_num)
pad_y <- 0.25

# complete grid to keep tile geometry identical across facets
all_AB  <- factor(c("0","5","50","500"), levels = c("0","5","50","500"))
all_Evo <- levels(droplevels(hf_dup$Evo_fac))

hf_plot_full <- expand_grid(
  Evo_fac = factor(all_Evo, levels = all_Evo),
  AB_fac  = factor(all_AB, levels = levels(all_AB)),
  row_label_plain = factor(y_levels, levels = y_levels)
) %>%
  left_join(
    hf_dup %>% select(Evo_fac, AB_fac, row_label_plain, COUNT),
    by = c("Evo_fac","AB_fac","row_label_plain")
  ) %>%
  mutate(COUNT = replace_na(COUNT, 0L),
         row_label_plain = as.character(row_label_plain)) %>%
  left_join(y_num_tbl, by = "row_label_plain")

AMP_COL <- "#6D2682"
hit_vals  <- sort(unique(hf_plot_full$COUNT))
nz_vals   <- setdiff(hit_vals, 0)
fill_vals <- alpha(AMP_COL, seq(0.35, 1, length.out = max(1, length(nz_vals))))
names(fill_vals) <- as.character(nz_vals)
fill_vals <- c("0" = "#FFFFFF", fill_vals)

y_labels_expr <- parse(text = unname(y_expr_map[y_levels]))

panel_heat <- ggplot(
  hf_plot_full,
  aes(x = AB_fac, y = y_num, fill = factor(COUNT, levels = names(fill_vals)))
) +
  geom_tile(colour = "white", linewidth = 0.15) +
  scale_fill_manual(
    values = fill_vals,
    breaks = names(fill_vals)[names(fill_vals) != "0"],
    name   = "Replicates\nhit (dup.)"
  ) +
  scale_x_discrete(
    labels = c("0"="0","5"="5","50"="50","500"="500"),
    drop   = FALSE,
    expand = expansion(add = c(0, 0))
  ) +
  scale_y_continuous(
    breaks = y_num_tbl$y_num,
    labels = y_labels_expr,
    expand = expansion(add = c(0, 0))
  ) +
  facet_grid(. ~ Evo_fac) +
  labs(x = "Ampicillin pre-pulse priming (\u00B5g ml\u207B\u00B9)", y = NULL) +
  coord_fixed(
    ratio = 0.30,
    ylim  = c(1 - pad_y, y_max + pad_y),
    clip  = "on"
  ) +
  theme_minimal(base_size = 9) +
  theme(
    panel.grid       = element_blank(),
    panel.border     = element_rect(colour = "black", fill = NA, linewidth = 0.3),
    axis.text.x      = element_text(angle = 0, vjust = 0.5, hjust = 0.5, size = 8),
    axis.title.x     = element_text(size = 9, face = "bold", margin = margin(t = 6)),
    axis.text.y      = element_text(size = 5.5, margin = margin(r = -10)),
    strip.placement  = "outside",
    strip.background = element_blank(),
    strip.text.x     = element_text(face = "bold", size = 8, lineheight = 0.95),
    legend.position  = "right",
    plot.margin      = margin(t = 12, r = 0, b = 4, l = 2),
    panel.spacing.x  = unit(0.05, "lines")
  )

legend_source <- panel_heat +
  theme(
    legend.position   = "right",
    legend.title      = element_text(size = 9),
    legend.text       = element_text(size = 8),
    legend.margin     = margin(0, 0, 0, 0),
    legend.box.margin = margin(0, 0, 0, 0)
  ) +
  guides(fill = guide_legend(ncol = 1, byrow = TRUE))

legend_heat <- cowplot::get_legend(legend_source)

final_plot <- cowplot::plot_grid(
  panel_heat + theme(legend.position = "none"),
  NULL,
  legend_heat,
  nrow = 1,
  rel_widths = c(1, 0.02, 0.18),
  align = "h",
  axis = "tb"
)

pdf_dev <- if (capabilities("cairo")) cairo_pdf else "pdf"

ggsave(
  filename = file.path(fig_dir, "ED_Fig6.pdf"),
  plot     = final_plot,
  width    = 5.5,
  height   = 5.5,
  units    = "in",
  device   = pdf_dev
)

message("Final figure: ", normalizePath(file.path(fig_dir, "ED_Fig6.pdf")))


=== WGS duplicated genes per species (after QC) ===
# A tibble: 4 × 2
  Species    duplicated_genes
  <chr>                 <int>
1 HAMBI_1972               31
2 HAMBI_1896               20
3 HAMBI_2659               14
4 HAMBI_1977                2


Final figure: /Users/cairns/Documents/Academic/Biology/2026/Projects/APEX/analysis/Downstream_analysis_polished/figs/ED_Fig6.pdf

